**This notebook is to create a table using system id**

In [0]:
# #import libraries
# import yaml

In [0]:
# # Notebook parameters
# dbutils.widgets.text("mode", "process")
# dbutils.widgets.text("yaml_path", "")
# dbutils.widgets.text("yaml_base_path", "")
# dbutils.widgets.text("yaml_file_name", "")
# mode = dbutils.widgets.get("mode")
# # MODE 1: Discover YAML files
# if mode == "discover":
#     yaml_path = dbutils.widgets.get("yaml_path").rstrip("/") + "/"
#     files = dbutils.fs.ls(yaml_path)
#     yaml_files = [
#         {"yaml_base_path": yaml_path, "yaml_file_name": file.name}
#         for file in files
#         if file.name.lower().endswith((".yaml", ".yml"))
#     ]
#     if not yaml_files:
#         raise ValueError(f"No YAML files found in {yaml_path}")
#     print(f"Total YAML files found: {len(yaml_files)}")
#     dbutils.jobs.taskValues.set(key="yaml_files", value=yaml_files)
# # MODE 2: Process individual YAML file
# elif mode == "process":
#     yaml_base_path = dbutils.widgets.get("yaml_base_path")
#     yaml_file_name = dbutils.widgets.get("yaml_file_name")
#     yaml_path = (f"{yaml_base_path.rstrip('/')}/{yaml_file_name}")
#     print(f"Processing YAML: {yaml_path}")

In [0]:
import yaml

# Notebook parameters
dbutils.widgets.text("mode", "process")
dbutils.widgets.text("yaml_path", "")
dbutils.widgets.text("yaml_base_path", "")
dbutils.widgets.text("yaml_file_name", "")
mode = dbutils.widgets.get("mode")
# MODE 1: Discover YAML files from all subfolders
if mode == "discover":
    yaml_path = dbutils.widgets.get("yaml_path").rstrip("/") + "/"
    yaml_files = []
    # Get all folders: bronze, control, gold, silver
    folders = dbutils.fs.ls(yaml_path)
    for folder in folders:
        if folder.isDir():
            print(f"Scanning folder: {folder.path}")
            # Get YAML files inside each folder
            files = dbutils.fs.ls(folder.path)
            for file in files:
                if file.name.lower().endswith((".yaml", ".yml")):
                    yaml_files.append({"yaml_base_path": folder.path, "yaml_file_name": file.name})
    if not yaml_files:
        raise ValueError(f"No YAML files found in {yaml_path}")
    print(f"Total YAML files found: {len(yaml_files)}")
    # Pass YAML file list to Databricks For each
    dbutils.jobs.taskValues.set(key="yaml_files", value=yaml_files)
# MODE 2: Process individual YAML file
elif mode == "process":
    yaml_base_path = dbutils.widgets.get("yaml_base_path")
    yaml_file_name = dbutils.widgets.get("yaml_file_name")
    if not yaml_base_path or not yaml_file_name:
        raise ValueError("yaml_base_path and yaml_file_name are required")
    yaml_path = f"{yaml_base_path.rstrip('/')}/{yaml_file_name}"
    print(f"Processing YAML: {yaml_path}")
    # Read YAML file
    yaml_content = dbutils.fs.head(yaml_path, 10000)
    yaml_config = yaml.safe_load(yaml_content)

    # Extract table details
    table_name = yaml_config["table"]["name"]
    table_schema_name = f"customer_billing.{yaml_config['table']['schema_name']}"
    table_columns = yaml_config["table"]["columns"]
    table_column_name = ",\n ".join([
        f'{table_column["name"]} {table_column["data_type"]}'
        for table_column in table_columns
    ])

    # Create schema
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {table_schema_name}")
    # Create table
    create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS {table_schema_name}.{table_name}
        ({table_column_name})
        USING DELTA
    """
    full_table_name = f"{table_schema_name}.{table_name}"
    if spark.catalog.tableExists(full_table_name):
        print(f"Table already exists: {full_table_name}")
    else:
        spark.sql(create_table_sql)
        print(f"Table created successfully: {full_table_name}")
else:
    raise ValueError(f"Invalid mode: {mode}")

In [0]:
# yaml_files = dbutils.fs.head(yaml_path, 10000)
# yaml_config =  yaml.safe_load(yaml_files)
# #read yaml file
# # with open(yaml_path, "r") as yaml_files:
# #     yaml_config = yaml.safe_load(yaml_files)
# # print(yaml_config)
# # print(yaml_config["table"])
# table_name = yaml_config["table"]["name"]
# table_schema_name = f"customer_billing.{yaml_config['table']['schema_name']}"
# table_columns = yaml_config["table"]["columns"]
# table_column_name = ",\n ".join([f'{table_column["name"]} {table_column["data_type"]}' for table_column in table_columns])
# # print(table_column_name)

In [0]:
# #Create schema if it does not exist
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {table_schema_name}")

# #Prepare CREATE TABLE statement
# create_table_sql = f"""CREATE TABLE IF NOT EXISTS {table_schema_name}.{table_name} ({table_column_name}) USING DELTA"""

# #Prepare complete table name
# full_table_name = f"{table_schema_name}.{table_name}"

# # Check whether table already exists
# if spark.catalog.tableExists(full_table_name):
#     print(f"Table already exists: {full_table_name}")
# else:
#     # Create table only when it does not exist
#     spark.sql(create_table_sql)
#     print(f"Table created successfully: {full_table_name}")

In [0]:
# create_table_sql = f"""
# create table if not exists {table_schema_name}.{table_name}({table_column_name})
# """
# # print(create_table_sql)
# spark.sql(create_table_sql)